In [15]:
import pyupbit
from core.upbit import Upbit
from loader.upbit_realtimedata_loader import UpbitRealtimeDataLoader
from processing.moving_average import MovingAverageProcessing
from processing.rsi import RSIProcessing
from processing.high_point_scoring import HighPointScoringProcessing
from processing.low_point_scoring import LowPointScoringProcessing
from processing.filtering_high_points import GetHighPoints
from processing.filtering_low_points import GetLowPoints
from processing.get_trend_section import  GetTrendSections
from processing.ma_200_rising import MA200RisingProcessing


from visualization.basic_price_rsi_visualization import BasicPriceWithRsiVisualization

import importlib
import common.common as common_module
importlib.reload(common_module)
from common.common import *

#여기가 최종 완성 구간이네

#60분일 때 
#interval_base = "minute60"
#load_count = 240
#score_band_list = [12] #12시간에 한번씩 채점
#threshold = 0.90
#indecreasing_count_set=[3,0] #3번 계속 상승구간만 체크하고 허용을 0번함

figure_to_jpg_dict = {} #저장한 이미지와 그 경로를 가진 dict 를 생성하고 아래카카오에서 이 정보를 기반으로 메시지를 전송함


#interval_base = "minute240"
interval_base = "day1"
load_count = 400
score_band_list = [100] #5일중 제일 높은 구간 채점
threshold = 0.85
indecreasing_count_set=[3,1]
ma_200_rising_min_consecutive_true = 2  # ma_200_rising 연속 True 최소 개수
aligned_in_order_window_size = 8  # 최근 8구간 모두 aligned_in_order=True 조건

target_coin_name_list = []

print("a")
## 업비트 전체 뒤져보기
tickers = pyupbit.get_tickers('KRW')

#코인별 확인
for idx, one_coin in enumerate(tickers):

    #if(idx==3):
    #    break
    
    #if one_coin not in base_date_dict.keys(): #최근 변곡점이 아닌 애들이라면 아예 하지 않음
    #    continue
    
    if(one_coin not in ['KRW-CARV']):
        continue
    
    print(f"{one_coin} 시작")
    
    #RAW 데이터 로드
    upbit = Upbit()
    upbit.set_loader(UpbitRealtimeDataLoader(one_coin,interval_base,load_count))
    upbit.load()
    #upbit.data = upbit.data.loc[upbit.data['timestamp_kst'] <= '2025-03-27 00:00:00']
    #upbit.data = upbit.data.loc[upbit.data['timestamp_kst'] <= '2025-03-26 23:00:00']
    
    #지표 추가
    indicator_list = [MovingAverageProcessing(), RSIProcessing(),HighPointScoringProcessing(score_band_list),LowPointScoringProcessing(score_band_list),MA200RisingProcessing()]
    upbit.add_sub_indicator(indicator_list)
    
    #고점, 저점 찾기
    upbit.generate_high_low_data(GetHighPoints(upbit.data.loc[upbit.data['high_score']!=0], 'high_score', threshold), 
                                GetLowPoints(upbit.data.loc[upbit.data['low_score']!=0], 'low_score', threshold)
                                )
    
    #고.저점 상승 추세구간 획득
    upbit.set_processor(GetTrendSections('high','increasing',indecreasing_count_set[0],indecreasing_count_set[1]))
    high_point_increasing_trend_section_upbit = upbit.get_key_points(upbit.high_point_df)
    upbit.set_processor(GetTrendSections('high','decreasing',indecreasing_count_set[0],indecreasing_count_set[1]))
    high_point_decreasing_trend_section_upbit = upbit.get_key_points(upbit.high_point_df)
    
    upbit.set_processor(GetTrendSections('low','increasing',indecreasing_count_set[0],indecreasing_count_set[1]))
    low_point_increasing_trend_section_upbit = upbit.get_key_points(upbit.low_point_df)
    upbit.set_processor(GetTrendSections('low','decreasing',indecreasing_count_set[0],indecreasing_count_set[1]))
    low_point_decreasing_trend_section_upbit = upbit.get_key_points(upbit.low_point_df)
    
    #저점고점 함께 상승하는 구간 찾기
    high_point_group_start_end_upbit = []
    for i in high_point_increasing_trend_section_upbit:
        high_point_group_start_end_upbit.append([i[0],i[-1]])
    
    
    low_point_group_start_end_upbit = []
    for i in low_point_increasing_trend_section_upbit:
        low_point_group_start_end_upbit.append([i[0],i[-1]])

    
    #고점은 낮아지는 구간 찾기
    #high_point_group_start_end_upbit = []
    #for i in high_point_decreasing_trend_section_upbit:
    #    high_point_group_start_end_upbit.append([i[0],i[-1]])
    
    
    
    high_low_increasing_trend_section_upbit, high_low_list_dict = find_overlapping_intervals(high_point_group_start_end_upbit, low_point_group_start_end_upbit)

    # upbit.low_point_df의 low 값이 3번 연속 하락 후 1번 상승 시작 지점 찾기
    low_decline_then_rise_points = []
    if len(upbit.low_point_df) > 3:
        low_values = upbit.low_point_df['low'].values
        low_indices = upbit.low_point_df.index.tolist()

        for i in range(len(low_values) - 3):
            if low_values[i] > low_values[i + 1] > low_values[i + 2] and low_values[i + 2] < low_values[i + 3]:
                low_decline_then_rise_points.append(low_indices[i + 3])
    
    
    #일단 주목해야할 코인들의 리스트를 받은
    target_coin_name_list.append(one_coin)
    
    
    #출력
    draw_instance = BasicPriceWithRsiVisualization()
    
    draw_instance.set_data(upbit.data, upbit.high_point_df, upbit.low_point_df)
    draw_instance.make_figure(title=one_coin)
    
    figure = draw_instance.get_figure()
    
    #add_vrect_to_main_figure(figure, upbit.data, 'red', high_low_increasing_trend_section_upbit) #중첩된 구간만 그리기
    #add_vrect_to_main_figure(figure, upbit.data, 'red', high_point_group_start_end_upbit) #고점 그래프는 빨간색
    #add_vrect_to_main_figure(figure, upbit.data, 'blue', low_point_group_start_end_upbit) #고점 그래프는 파란색
    #add_vrect_to_main_figure(figure, upbit.data, '#b8860b', dark_yellow_group_start_end_upbit, show_interval_label=True, label_prefix='번 구간', start_label_index=1, show_interval_length=True, interval_length_separator=' : ') #조건 충족 구간은 진한 노란색 + 구간 번호/길이

    if len(low_decline_then_rise_points) > 0:
        add_vline_to_main_figure(
            figure,
            upbit.low_point_df,
            'red',
            low_decline_then_rise_points,
            label_text='일봉 저점 상승 시작',
        )

    print(f"{one_coin} 출력")
    draw_instance.visualize()
    
    #파일이름으로 저장함
    
    #jpg_file_name = JPG_DIRECTORY+'/'+generate_jpg_file_name(one_coin)
    #print(f"파일 저장 :{jpg_file_name}")
    #figure.write_image(jpg_file_name, format="jpg")
    #print("완료")

    
    
    
    #break

a
KRW-CARV 시작
업비트에서 KRW-CARV 가격을 최신부터day1 간격으로 400개 로드 합니다.
가장 높은 점수:53
가장 높은 점수:100
정형화된 기준값은 : 0.16346153846153846
정형화된 기준값은 : 0.19090909090909058
KRW-CARV 출력


c:\Users\baram\dongseop\dongseop\processing\filtering_high_points.py:21: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

c:\Users\baram\dongseop\dongseop\processing\filtering_high_points.py:29: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

c:\Users\baram\dongseop\dongseop\processing\filtering_high_points.py:31: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://panda

In [12]:
low_decline_then_rise_points

[397]

In [3]:
upbit.low_point_df

,timestamp_kst,open,low,high,close,ma_20,ma_60,ma_200,aligned_in_order,RSI,high_score,low_score,ma_200_rising,normalized_value_low,origin_low_score,low_for_graph,increasing,decreasing
128,2025-06-27 09:00:00,308.3,294.6,316.9,315.8,376.690,471.713333,497.446512,False,23.329208,0,100.0,False,1.000000,100.0,294.6,False,False
245,2025-10-22 09:00:00,232.0,215.0,235.0,221.0,306.550,379.650000,424.308500,False,7.853403,0,29.0,False,0.282828,29.0,215.0,False,True
275,2025-11-21 09:00:00,213.0,173.0,214.0,193.0,258.300,292.116667,389.042500,False,13.286713,0,39.0,False,0.383838,39.0,173.0,False,True
352,2026-02-06 09:00:00,86.5,79.6,96.0,94.8,124.695,169.965000,290.038500,False,25.913242,0,100.0,False,1.000000,100.0,79.6,False,True
397,2026-03-23 09:00:00,83.4,80.4,83.4,83.2,85.645,97.155000,215.221500,False,51.666667,0,45.0,False,0.444444,45.0,80.4,True,False


In [1]:
import pyupbit

In [ ]:
import pandas as pd

import pandas as pd

pd.set_option("display.max_rows", None)         # 모든 행
pd.set_option("display.max_columns", None)      # 모든 열
pd.set_option("display.width", None)            # 줄바꿈 폭 제한 해제
pd.set_option("display.max_colwidth", None)     # 컬럼 내용 생략(...) 방지
pd.set_option("display.expand_frame_repr", False)  # 가로로 펼쳐서 보기
upbit.data[pd.to_datetime(upbit.data["timestamp_kst"]).dt.strftime("%Y-%m") == "2026-03"]